# 1.14 — Calling Instance Methods 🧋

### AP CSA · Unit 1: Using Objects and Methods
**Boba Cafe Series · Lesson 14**

---

> **Setup note:** every code cell runs on the **IJava kernel** (Java 17+). Check that the kernel picker says *Java*. Each cell declares a class and then calls it with `ClassName.main(null);`.

## The dot you've never had to explain

`scan.next()`. `scan.nextInt()`. You've written the dot between an object and a method name in **every single notebook since Lesson 1.4**, and it always just worked, so there was never a reason to stop and ask what that dot was actually doing.

Time to ask. Today closes out two loose threads at once: what the dot operator really means, and the thing Lesson 1.12 flagged and then made you wait for — what happens when you call a method on an object that isn't there.

### What you'll be able to do by the end

| # | Objective | CED reference |
|---|---|---|
| 1 | Use the **dot operator** to call an instance method on an object | 1.14.A |
| 2 | Explain how an **instance method** differs from the `static` class methods of Lesson 1.10 | 1.14.A |
| 3 | Explain why the same instance method can give different results on different objects | 1.14.A |
| 4 | Predict and explain a `NullPointerException` | 1.14.A |
| 5 | Recognize when code is at risk of calling a method on a `null` reference | 1.14.A |

---

## Part 1 — The dot operator

> **The dot operator, `object.method(arguments)`, calls a method that belongs to a specific object.**

This is the instance-method counterpart to `ClassName.method(...)` from Lesson 1.10. The two look almost identical and mean genuinely different things:

```
   Math.sqrt(64)              scan.next()
   ^^^^ ^^^^^^^^^              ^^^^ ^^^^^^
    |       |                   |      |
    |       └─ the method       |      └─ the method
    └─ a CLASS name             └─ an OBJECT reference
      (Lesson 1.10)               (this lesson)
```

`Math.sqrt(64)` asks the `Math` **class itself** to compute something — no object exists anywhere in that call. `scan.next()` asks **one specific `Scanner` object** to hand back its next token — and that object's own internal position is what makes the answer meaningful. You've been calling instance methods correctly this whole series; today you get the name for it.

In [ ]:
import java.util.Scanner;

public class TheDotOperator {
    public static void main(String[] args) {
        Scanner order = new Scanner("Taro Milk Tea");
        String flavor = "Brown Sugar";

        // object.method(arguments) -- the dot operator, calling INSTANCE methods
        System.out.println("order.next():         " + order.next());
        System.out.println("flavor.length():      " + flavor.length());
        System.out.println("flavor.toUpperCase(): " + flavor.toUpperCase());
    }
}

TheDotOperator.main(null);

---

## Part 2 — Instance methods work on that object's own data

This is the reason instance methods exist at all, and it's the same idea you already proved in Lessons 1.12 and 1.13: **every object carries its own copy of the data described by its class**, and an instance method operates on *that specific object's* copy — never on some shared, class-wide version.

Call `.length()` on two different `String` objects and you get two different, correct answers, because each string is its own object with its own characters.

In [ ]:
public class SameMethodDifferentObjects {
    public static void main(String[] args) {
        String short_ = "Taro";
        String long_ = "Brown Sugar Milk Tea";

        // The SAME instance method, .length(), called on TWO DIFFERENT objects.
        System.out.println("short_.length(): " + short_.length());
        System.out.println("long_.length():  " + long_.length());
    }
}

SameMethodDifferentObjects.main(null);

`.length()` isn't a formula that always returns the same thing, the way you might expect from a static utility method. It's a question **the object answers about itself**, and different objects answer it differently — because the answer genuinely depends on which object you asked.

This is the same principle behind the independent `Scanner` objects from Lesson 1.12: `scanA.next()` and `scanB.next()` gave different results not because `next()` behaves randomly, but because each object was tracking its *own* separate state.

---

## Part 3 — Class methods vs. instance methods, side by side

You now have both halves of method-calling in Java. Here's the full comparison, worth memorizing as a pair.

| | Class method (`static`) — Lesson 1.10 | Instance method — this lesson |
|---|---|---|
| Called as | `ClassName.method(...)` | `object.method(...)` |
| Needs an object first? | No | **Yes** — must exist before you can call anything on it |
| Operates on | Nothing object-specific | That **one object's** own data |
| Example | `Math.sqrt(64)` | `"Taro".length()` |
| Example | `Integer.parseInt("5")` | `scan.next()` |

The tell is almost always right there in the syntax: a capitalized word before the dot is usually a **class**, and a lowercase variable before the dot is usually an **object**. That's not a hard rule — classes and variables can technically be named however you like — but Java naming conventions (Lesson 1.6) make it a reliable signal in practice.

In [ ]:
public class BothKindsTogether {
    public static void main(String[] args) {
        String drink = "Taro Milk Tea";

        // CLASS method: static, called on the Math CLASS, no object involved
        double sqrtOfLength = Math.sqrt(drink.length());

        // INSTANCE method: called on drink, the specific String OBJECT
        int len = drink.length();

        System.out.println("Length of \"" + drink + "\": " + len);
        System.out.println("Square root of that length: " + sqrtOfLength);
    }
}

BothKindsTogether.main(null);

`drink.length()` appears **twice** in that program, and both times it's an instance method call on the same object — but the result gets used differently each time: once stored, once passed straight into another method call as an argument. Either way is legal, exactly per the return-value rules from Lesson 1.9.

---

## Part 4 — `NullPointerException`

Lesson 1.12 told you that printing a `null` reference is completely safe — it just prints the word `null`. **Calling a method on one is not.**

> **Calling an instance method on a `null` reference throws a `NullPointerException`.** There is no object there to answer the question, and Java has no fallback behavior — it crashes immediately.

**Heads up: the next cell is supposed to fail.**

In [ ]:
public class CallOnNull {
    public static void main(String[] args) {
        String flavor = null;

        System.out.println("About to call a method on a null reference...");
        int len = flavor.length();       // there is no object here to ask!
        System.out.println("This line never runs: " + len);
    }
}

CallOnNull.main(null);

`NullPointerException` — Java's way of saying *"you asked an object to do something, but that reference doesn't point to any object at all."* Notice the first `println` **did** print, and the program crashed on the very next line. That's the run-time-error fingerprint from Lesson 1.1: partial output, then a hard stop.

> The exact wording of a `NullPointerException` message can vary slightly depending on how the code was compiled, but it always names the method that failed and confirms the reference was `null`.

The same rule applies to every reference type, not just `String`:

In [ ]:
import java.util.Scanner;

public class CallOnNullScanner {
    public static void main(String[] args) {
        Scanner order = null;
        System.out.println("Calling next() on a null Scanner...");
        String token = order.next();
    }
}

CallOnNullScanner.main(null);

Same category of crash, different class — `NullPointerException` doesn't care which reference type was involved. It only cares that *something* tried to call a method through a reference that pointed at nothing.

---

## Part 5 — Spotting the risk before it crashes

This is the part the AP exam likes to hide inside otherwise-normal-looking code: a method that returns an object **sometimes**, and `null` other times. The bug isn't in the method — it's in code that calls something on the result without checking first.

In [ ]:
public class LookupDrink {

    /**
     * Looks up a drink's price by name.
     * Precondition:  drinkName is not null
     * Postcondition: returns the price as a String if the drink is on the menu,
     *                or null if the drink is NOT on the menu
     */
    public static String priceOf(String drinkName) {
        if (drinkName.equals("Taro Milk Tea")) return "$5.75";
        if (drinkName.equals("Matcha Latte"))   return "$6.50";
        return null;    // not on the menu -- there's simply nothing to return
    }

    public static void main(String[] args) {
        String price = priceOf("Taro Milk Tea");
        System.out.println("Taro price: " + price);

        String missingPrice = priceOf("Bubble Coffee");    // not on the menu
        System.out.println("Length of missing price string: " + missingPrice.length());
    }
}

LookupDrink.main(null);

`priceOf("Taro Milk Tea")` worked fine, which is exactly what makes this trap dangerous: **the first call succeeding gives no guarantee the second one will.** `"Bubble Coffee"` isn't on the menu, `priceOf` correctly returned `null` per its own documented postcondition (Lesson 1.8), and the caller called `.length()` on that `null` without checking — crash.

This is precisely the situation the caution note in Lesson 1.12 was pointing toward. The fix isn't inside `priceOf` — it did exactly what it promised. The fix belongs to whoever calls it: **check whether a result could be `null` before calling a method on it**, especially anytime a method's documentation says it might return `null`.

---

# Practice: Debugging the Order Screen

Four tasks, in order.

---

## Hack 1 — Class method or instance method?

For each call, say whether it's a class method call or an instance method call, and how you can tell. Double-click to edit.

| Call | Class or instance? | How you can tell |
|---|---|---|
| `Math.pow(2, 5)` | | |
| `scan.nextInt()` | | |
| `"Taro".toUpperCase()` | | |
| `Integer.MAX_VALUE` | | |
| `drink.length()` | | |

<details>
<summary><b>Check your answers</b></summary>

| Call | Class or instance? | How you can tell |
|---|---|---|
| `Math.pow(2, 5)` | Class | Called on `Math`, a class name, no object involved |
| `scan.nextInt()` | Instance | Called on `scan`, a specific `Scanner` object |
| `"Taro".toUpperCase()` | Instance | Called directly on a `String` object (the literal itself) |
| `Integer.MAX_VALUE` | Neither — it's an attribute | No parentheses at all, so it's not a method call; it's a stored constant, read via the class (Lesson 1.7) |
| `drink.length()` | Instance | Called on `drink`, a specific `String` object |

`Integer.MAX_VALUE` is there on purpose — a good reminder that not everything with a dot in it is a method call at all.
</details>

---

## Hack 2 — Predict the crash

The cell below prints some lines successfully, then crashes. **Before running it**, predict exactly which line will be the last one printed, and why.

**Your prediction:**

In [ ]:
public class PredictTheCrash {
    public static void main(String[] args) {
        String customerA = "Riley";
        String customerB = null;

        System.out.println("A: " + customerA);
        System.out.println("A's initial: " + customerA.substring(0, 1));
        System.out.println("B: " + customerB);
        System.out.println("B's initial: " + customerB.substring(0, 1));
        System.out.println("This line never runs.");
    }
}

PredictTheCrash.main(null);

<details>
<summary><b>Check your answer</b></summary>

The last line to print successfully is `"B: null"`.

Printing `customerB` — a `null` reference — is completely safe, per Lesson 1.12; it just prints the word `null`. The crash happens on the **next** line, where `customerB.substring(0, 1)` tries to *call a method* on that same `null` reference. That's the distinction this whole lesson rests on: printing `null` is fine, calling a method on `null` is not.
</details>

---

## Hack 3 — Fix the lookup

The cell below is a rewritten version of Part 5's `priceOf` example, applied to a loyalty-status lookup. It crashes for any customer not in the program. Run it, then fix `main` so it handles a missing customer gracefully — **without modifying `statusOf` itself**, since it's already correctly documented and behaving exactly as promised.

Your fixed version should print `"Status: not found"` instead of crashing when the customer isn't recognized.

In [ ]:
public class BrokenLookup {

    /**
     * Looks up a customer's loyalty status.
     * Postcondition: returns their status if known, or null if the customer is not found
     */
    public static String statusOf(String customerName) {
        if (customerName.equals("Riley"))  return "Gold";
        if (customerName.equals("Jordan")) return "Silver";
        return null;
    }

    public static void main(String[] args) {
        String status = statusOf("Sam");             // not in the system
        System.out.println("Status: " + status.toUpperCase());
    }
}

BrokenLookup.main(null);

**Your fix** (write your corrected `main` method below, in a comment, or restructure the cell above):

<details>
<summary><b>One possible solution</b></summary>

```java
public static void main(String[] args) {
    String status = statusOf("Sam");

    if (status == null) {
        System.out.println("Status: not found");
    } else {
        System.out.println("Status: " + status.toUpperCase());
    }
}
```

`status == null` uses `==` exactly as described in Lesson 1.13 — checking whether the reference points to no object at all, before ever attempting to call a method on it. This `if`/`else` shape (formally taught in Unit 2) is the standard defense against `NullPointerException` whenever a method's documentation admits it might return `null`.
</details>

---

## Hack 4 — Build a safe menu lookup

Write your own version of `priceOf` from Part 5, for a **topping** lookup instead of a drink lookup:

1. A method `toppingPrice(String toppingName)` that returns `"$0.75"` for `"Pearls"`, `"$0.50"` for `"Jelly"`, and `null` for anything else — documented with a Javadoc comment including a postcondition, matching the style from Lesson 1.8.
2. A `main` method that calls `toppingPrice` for **three** toppings: one that exists, one that doesn't, and your choice of a third.
3. For each result, **check for `null` before calling any method on it** — print the price if found, or `"Not available"` if not.

In [ ]:
public class MyToppingLookup {

    // TODO 1: write toppingPrice(String toppingName) with a full Javadoc comment

    public static void main(String[] args) {

        // TODO 2: call toppingPrice for three toppings

        // TODO 3: for each result, check for null before using it

    }
}

MyToppingLookup.main(null);

<details>
<summary><b>One possible solution</b></summary>

```java
public class MyToppingLookup {

    /**
     * Looks up the price of a topping.
     * Precondition:  toppingName is not null
     * Postcondition: returns the price as a String if the topping is available,
     *                or null if it is not on the menu
     * @param toppingName the topping being looked up
     * @return the price, or null if not found
     */
    public static String toppingPrice(String toppingName) {
        if (toppingName.equals("Pearls")) return "$0.75";
        if (toppingName.equals("Jelly"))  return "$0.50";
        return null;
    }

    public static void main(String[] args) {
        String[] toppings = {"Pearls", "Popcorn Chicken", "Jelly"};

        for (String name : toppings) {
            String price = toppingPrice(name);
            if (price == null) {
                System.out.println(name + ": Not available");
            } else {
                System.out.println(name + ": " + price);
            }
        }
    }
}

MyToppingLookup.main(null);
```

> This solution uses a `for-each` loop over an array to avoid repeating the same three lines three times — both arrays and this loop form are ahead of where the course has formally gotten (Units 2 and 4). Three separate calls, each with its own `if`/`else`, work exactly as well if you'd rather stick to material from this notebook alone.

Output:

```
Pearls: $0.75
Popcorn Chicken: Not available
Jelly: $0.50
```

Every result gets checked for `null` before anything is called on it — zero risk of a `NullPointerException`, no matter what gets looked up.
</details>

---

# Self-Check: AP-style questions

**1.** Which correctly describes the dot operator?

&nbsp;&nbsp;(A) It calls a method belonging to a specific object
&nbsp;&nbsp;(B) It is only used with `static` methods
&nbsp;&nbsp;(C) It separates a package name from a class name
&nbsp;&nbsp;(D) It is required before every method call, including class methods

<details><summary>Answer</summary>

**(A)**. `object.method(...)` calls an instance method on a specific object. (B) describes `ClassName.method(...)` from Lesson 1.10, the opposite case.
</details>

---

**2.** Why can `"Taro".length()` and `"Matcha Latte".length()` return different values, even though they call the identical method?

&nbsp;&nbsp;(A) `.length()` is random
&nbsp;&nbsp;(B) Each `String` object has its own characters, and `.length()` reports on that specific object's data
&nbsp;&nbsp;(C) One of the two calls is actually a class method
&nbsp;&nbsp;(D) Only one of the calls is legal Java

<details><summary>Answer</summary>

**(B)**. An instance method operates on the object it's called on. Two different `String` objects hold two different sequences of characters, so the same method legitimately reports two different results.
</details>

---

**3.** What is printed?

```java
String name = null;
System.out.println("Name: " + name);
System.out.println("Length: " + name.length());
```

&nbsp;&nbsp;(A) `Name: null` then `Length: 0`
&nbsp;&nbsp;(B) `Name: null`, then a `NullPointerException`
&nbsp;&nbsp;(C) A compile-time error on the first line
&nbsp;&nbsp;(D) Nothing prints at all

<details><summary>Answer</summary>

**(B)**. Printing `null` is safe and prints the literal word `null`. Calling `.length()` on that same `null` reference is a different operation entirely — it requires an actual object to answer the question, and there isn't one, so it throws a `NullPointerException`.
</details>

---

**4.** A method's documentation states: `Postcondition: returns the matching record, or null if none is found.` What should a caller do before using the returned value?

&nbsp;&nbsp;(A) Nothing — the method guarantees a valid object every time
&nbsp;&nbsp;(B) Check whether the result is `null` before calling any method on it
&nbsp;&nbsp;(C) Wrap the call in a `try`/`catch` block, since that is the only valid approach
&nbsp;&nbsp;(D) Call the method twice to confirm the result is consistent

<details><summary>Answer</summary>

**(B)**. The documented postcondition explicitly admits `null` is a possible result. A responsible caller checks for it — typically with `result == null` — before attempting to call any method on that reference.
</details>

---

**5.** Which of these is a class method call rather than an instance method call?

&nbsp;&nbsp;(A) `scanner.hasNext()`
&nbsp;&nbsp;(B) `"hello".toUpperCase()`
&nbsp;&nbsp;(C) `Integer.parseInt("42")`
&nbsp;&nbsp;(D) `order.next()`

<details><summary>Answer</summary>

**(C)**. `Integer.parseInt(...)` is called on the `Integer` class itself, with no object involved — a class method, exactly like `Math.sqrt` from Lesson 1.10. The other three are all called on a specific object.
</details>

---

**6.** What is the immediate cause of a `NullPointerException`?

&nbsp;&nbsp;(A) Declaring a reference variable without assigning it
&nbsp;&nbsp;(B) Comparing two objects with `==`
&nbsp;&nbsp;(C) Calling an instance method on a reference that currently points to no object
&nbsp;&nbsp;(D) Passing `null` as an argument to a method

<details><summary>Answer</summary>

**(C)**. The exception fires specifically when code attempts to call a method (or access a field) through a reference whose value is `null`. (A) is actually a *different* error — a compile-time "might not have been initialized" error from Lesson 1.12 — and (D) alone, without calling a method on the `null` value, causes no problem at all.
</details>

---

**7.** Which statement about `Math.sqrt(16)` versus `"boba".length()` is correct?

&nbsp;&nbsp;(A) Both are instance methods
&nbsp;&nbsp;(B) Both are class methods
&nbsp;&nbsp;(C) `Math.sqrt` is a class method; `"boba".length()` is an instance method
&nbsp;&nbsp;(D) `Math.sqrt` is an instance method; `"boba".length()` is a class method

<details><summary>Answer</summary>

**(C)**. `Math.sqrt` is called on the `Math` class with no object involved. `"boba".length()` is called directly on a specific `String` object — the literal `"boba"` itself — making it an instance method call.
</details>

---

# Closing time

### Vocabulary to know cold

| Term | One-line definition |
|---|---|
| Dot operator | `object.method(...)` — calls a method belonging to a specific object |
| Instance method | A method that operates on one particular object's own data |
| Class method | A `static` method called through the class name, with no object (Lesson 1.10) |
| `NullPointerException` | Thrown when an instance method is called on a reference that points to no object |

### The five things that will show up on the exam

1. `object.method(...)` calls an **instance** method; `ClassName.method(...)` calls a **class** method.
2. The same instance method can give different results on **different objects** — that's the whole point.
3. Printing `null` is safe. **Calling a method on `null` throws `NullPointerException`.**
4. A method documented as possibly returning `null` needs a check **before** anything is called on the result.
5. `NullPointerException` produces partial output then a crash — the run-time-error fingerprint from Lesson 1.1.

### Before you submit, check that you:

- [ ] Ran every code cell, including the two that fail on purpose
- [ ] Classified all five calls in Hack 1
- [ ] Predicted the exact crash point in Hack 2 **before** running it
- [ ] Fixed Hack 3 without modifying `statusOf`
- [ ] Built a fully null-safe `toppingPrice` lookup in Hack 4
- [ ] Attempted all seven self-check questions before revealing answers

### Next up

**1.15 — String Manipulation.** `.length()` and `.toUpperCase()` were just the preview. Next shift is the full `String` toolkit — `substring`, `indexOf`, `equals`, and why `==` on two `String` objects is the one place this series has told you three separate times to wait for the real explanation. That wait is over.

See you at the next shift 🧋